# Volume 14, Chapter 3: The Erdős–Kac theorem, sharply (Python companion)

Exercises first, solutions at the end. We compare the law of $\omega(n)$, $n\le10^7$, with the Sathe–Selberg formula, with $1+\mathrm{Poisson}(\lambda)$ and with the Gaussian; measure the Kolmogorov distance of the Erdős–Kac theorem against the Rényi–Turán rate; look at large deviations and at the second-order term of $\pi_2(x)$. The computer science application: **a segmented sieve** that computes $\omega(n)$ for all $n\le N$ using only the primes up to $\sqrt N$ and memory $O(\sqrt N)$ per segment. Requires `numpy` and `scipy`.

In [ ]:
import math, time
import numpy as np
from scipy.special import gamma, ndtr
from scipy.stats import poisson

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

N = 10**7
P = primes_upto(N)
omega = np.zeros(N + 1, dtype=np.int8)
for p in P: omega[p::p] += 1
omega = omega[1:]
Pf = P.astype(float)
def F(z): return math.exp(np.sum(np.log1p((z - 1) / Pf) + (z - 1) * np.log1p(-1 / Pf)))
def G(z): return F(z) / gamma(z + 1)

## Exercises

**P3.1 (★) The local law.** For $x=10^7$ tabulate $P_x(\omega(n)=k)$, the Sathe–Selberg main term $e^{-\lambda}\frac{\lambda^{k-1}}{(k-1)!}G(\frac{k-1}\lambda)$ (Theorem 3.2), the law of $1+\mathrm{Poisson}(\lambda)$ and the Gaussian density (Figure 3.1).

**P3.2 (★) The Rényi–Turán rate.** Compute the Kolmogorov distance $D(x)$ between the law of $(\omega(n)-\lambda)/\sqrt\lambda$, $n\le x$, and the standard Gaussian for $x=10^3,\dots,10^7$, and the product $\sqrt\lambda D(x)$ (Figure 3.2). How much of $D(x)$ is due to the mean being $\lambda+0.26$ rather than $\lambda$?

**P3.3 (★★) Large deviations.** Compute $-\frac1\lambda\log P_x(\omega(n)\ge\alpha\lambda)$ for $\alpha=1.5,2$, and $-\frac1\lambda\log P_x(\omega(n)\le\alpha\lambda)$ for $\alpha=\frac12$, at $x=10^5,10^6,10^7$ and compare with $Q^*(\alpha)=\alpha\log\alpha-\alpha+1$ (Corollary 3.4). Is the convergence visible?

**P3.4 (★★) The second term of $\pi_2$.** Compute $\beta=\gamma+\sum_p(\frac1{p-1}+\log(1-\frac1p))$ and compare $\frac{\pi_2(x)\log x}x-\log\log x$ with $\beta$ for $x=10^4,\dots,10^7$ (Exercise 3.1).

**P3.5 (★★) The mode.** Find the most frequent value of $\omega(n)$, $n\le x$, for $x=10^2,10^3,\dots,10^7$, and compare with $\lambda$ (Exercise 3.4).

**P3.6 (★★) Computer science: a segmented sieve.** Every $n\le N$ has at most one prime factor larger than $\sqrt N$. Compute $\omega(n)$ on a segment $[a,b)$ by dividing out the primes up to $\sqrt N$ and counting a leftover cofactor $>1$ as one more prime. Check against the full sieve for $N=10^7$ with segments of length $10^6$, compare the memory used, and estimate the time for $N=10^9$.

## Solutions

### P3.1

In [ ]:
x = N; lam = math.log(math.log(x))
h = np.bincount(omega, minlength=10)[:10] / x
print(f"lambda = {lam:.4f}")
print(" k   true    Sathe-Selberg  1+Poisson  Gauss")
for k in range(1, 10):
    ss = poisson.pmf(k - 1, lam) * G((k - 1) / lam)
    gs = math.exp(-(k - lam)**2 / (2 * lam)) / math.sqrt(2 * math.pi * lam)
    print(f"{k:2d}  {h[k]:.4f}   {ss:.4f}         {poisson.pmf(k-1, lam):.4f}     {gs:.4f}")

The Sathe–Selberg term improves on the shifted Poisson law at most $k$, but with $\lambda=2.78$ none of the three is accurate: the relative error term $k/\lambda^2$ of Theorem 3.2 is not small for any $x$ we can reach.

### P3.2

In [ ]:
def kolmogorov(x, shift=0.0):
    lam = math.log(math.log(x))
    c = np.cumsum(np.bincount(omega[:x], minlength=12)) / x
    d = 0.0
    for k in range(len(c)):
        u = (k - lam - shift) / math.sqrt(lam)
        d = max(d, abs(c[k] - ndtr(u)), abs((c[k - 1] if k else 0) - ndtr(u)))
    return d
for e in range(3, 8):
    x = 10**e; lam = math.log(math.log(x)); d = kolmogorov(x)
    print(f"x=10^{e}: D={d:.4f}, sqrt(lambda) D={math.sqrt(lam) * d:.4f}, mean of omega - lambda = {omega[:x].mean() - lam:.3f}, D after recentring = {kolmogorov(x, omega[:x].mean() - lam):.4f}")

$\sqrt\lambda D(x)$ stays near $0.42$ to $0.45$: the rate $1/\sqrt{\log\log x}$ is visible. Recentring at the true mean, which exceeds $\lambda$ by $0.19$ to $0.23$ here and tends to Mertens' constant $0.2615$ (Volume 5), reduces the distance; what remains is mostly the lattice effect, the jumps of size $\asymp1/\sqrt\lambda$ that no continuous law can follow (Exercise 3.3).

### P3.3

In [ ]:
Qs = lambda a: a * math.log(a) - a + 1
for e in [5, 6, 7]:
    x = 10**e; lam = math.log(math.log(x))
    row = []
    for a in [0.5, 1.5, 2.0]:
        p = (omega[:x] <= a * lam).mean() if a < 1 else (omega[:x] >= a * lam).mean()
        row.append(f"alpha={a}: {-math.log(p) / lam:.3f} (Q*={Qs(a):.3f})")
    print(f"x=10^{e}:", "  ".join(row))

The empirical exponents are of the right size but do not visibly converge: the correction in Corollary 3.4 is $O(\log\lambda/\lambda)$, and here $\lambda<3$, so the error term is as large as the main term. Also $\{\omega(n)\ge\alpha\lambda\}$ is a lattice event whose threshold jumps as $\lambda$ grows. Large deviations of $\omega(n)$ are a theorem about $x$ we will never see.

### P3.4

In [ ]:
beta = 0.5772156649015329 + np.sum(1 / (Pf - 1) + np.log1p(-1 / Pf))
print("beta =", beta, "(tail beyond 10^7 is about -1/(2*10^7 log 10^7), negligible)")
for e in range(4, 8):
    x = 10**e; L = math.log(x); lam = math.log(L)
    pi2 = np.count_nonzero(omega[:x] == 2)
    print(f"x=10^{e}: pi_2 L/x - lambda = {pi2 * L / x - lam:.4f}")

The values decrease towards $\beta=1.0346$, slowly, as an error $O(1/\lambda)$ permits: at $10^7$ the excess $0.27$ is about $0.75/\lambda$.

### P3.5

In [ ]:
for e in range(2, 8):
    x = 10**e
    print(f"x=10^{e}: mode = {np.bincount(omega[:x]).argmax()}, lambda = {math.log(math.log(x)):.3f}")

### P3.6

In [ ]:
def omega_segment(a, b, small_primes):
    n = np.arange(a, b, dtype=np.int64)
    rem = n.copy(); w = np.zeros(b - a, dtype=np.int8)
    for p in small_primes:
        st = (-a) % p
        if st >= b - a: continue
        w[st::p] += 1
        pk = p
        while pk < b:
            st2 = (-a) % pk
            if st2 < b - a: rem[st2::pk] //= p
            pk *= p
    w += (rem > 1)
    return w
t = time.time()
sp = primes_upto(int(math.isqrt(N)))
seg = 10**6
ok = True
for a in range(1, N + 1, seg):
    b = min(a + seg, N + 1)
    ok &= bool(np.array_equal(omega_segment(a, b, sp), omega[a - 1:b - 1]))
dt = time.time() - t
print("segmented sieve agrees with the full sieve:", ok, f" time {dt:.1f}s")
print("memory per segment: about", seg * 9 // 10**6, "MB, versus", N * 9 // 10**6, "MB for one pass")
sp9 = primes_upto(31623)
print(f"rough time for N=10^9: {dt * 100 * len(sp9) / len(sp):.0f}s (100 times more segments, {len(sp9)} small primes instead of {len(sp)})")

The segmented sieve is how the data of Figures 3.1 and 3.2 up to $10^8$ were computed. Memory is the binding constraint for full sieves: a byte per integer is a gigabyte at $10^9$. The segment needs only the $\pi(\sqrt N)$ small primes and a buffer, and segments can run in parallel.